# SiPM_Characterization_DC_Study

Dark count (LED spento, self-trigger): stessa catena del notebook `SiPM_Characterization_Gain_Study` (filtro, baseline per evento, gate stretto alla stessa distanza dalla salita, fit con le posizioni dei picchi libere).

Cambia il modello: qui ogni evento ha un fotoelettrone primario certo, l'impulso che ha fatto scattare il trigger, e i picchi successivi vengono dal crosstalk. Si fitta con la Borel, che ha lo stesso lambda della Poisson generalizzata del Gain: i due valori si confrontano direttamente.

In [ ]:
## === Import principali ===
import os
import re
import contextlib, gzip, shutil, tempfile

import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy.optimize import curve_fit


# ============================================================
# LETTURA DEI FILE DELLA DAQ  (da WCFastCheck)
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================

@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]


def leggi_canale(path, canale, max_eventi=None):
    """Matrice (eventi, campioni) di un canale, in ADC int16, e SamplingTime [ns].

    Nel formato v2 il dataset si legge in un colpo solo: con eventi_di, un
    evento alla volta, mezzo milione di eventi richiedono minuti invece di
    secondi. Il v1 non lo permette e passa da eventi_di.
    """
    with apri_run(path) as f:
        sampling_time_ns = float(f["/config"].attrs["SamplingTime"]) * 1e9
        canali_file, (i_ch,) = canali_di(f, [canale])
        g = f["/events"]
        if "waveforms" in g:
            ds = g["waveforms"]
            n = ds.shape[0] if max_eventi is None else min(max_eventi, ds.shape[0])
            dati = ds[:n]
        else:
            chiavi, leggi = eventi_di(f)
            chiavi = chiavi[:max_eventi] if max_eventi is not None else chiavi
            dati = np.stack([leggi(k) for k in chiavi]).astype(np.int16)
    n_campioni = dati.shape[1] // len(canali_file)
    wf = dati.reshape(dati.shape[0], len(canali_file), n_campioni)[:, i_ch, :]
    return np.ascontiguousarray(wf), sampling_time_ns


In [ ]:
# ============================================================
# CONFIG
# ============================================================

# Il notebook sta in tools/, i dati in data/ alla radice del repo
DATA_DIR = os.path.abspath(os.path.join("..", "data/10062026/SiPM_001"))

# File del dataset. Ogni run si identifica con quello che cambia nello scan,
# letto dal nome: in uno scan in tensione la label e' la tensione (51-5V ->
# "51.5"), in uno scan in ampiezza del LED e' l'ampiezza (3LEDampl -> "3").
# Se nella lista cambiano tutte e due, la label le contiene entrambe
# ("52V_3LED"). Le chiavi per run (RUN_DA_MOSTRARE, PICCHI_MANUALI, ...) usano
# questa label; tensione e ampiezza di ogni run restano in VBIAS e LED_AMPL,
# e il fit in tensione deve prendere Vbias da VBIAS, non dalla label.
RUN_FILES = [
    "Test_SiPM_52V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_52-5V_NOLED_SelfTrigger_0001_750Ms_40PT.h5.gz",
    "Test_SiPM_53V_NOLED_SelfTrigger_0001_750Ms_40PT.h5.gz",
    "Test_SiPM_53-5V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_54V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_54-5V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_55V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_55-5V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    "Test_SiPM_56V_NOLED_SelfTrigger_0002_750Ms_40PT.h5.gz",
    "Test_SiPM_56-5V_NOLED_SelfTrigger_0000_750Ms_40PT.h5.gz",
    # "Test_SiPM_57V_NOLED_SelfTrigger_0001_750Ms_40PT.h5.gz",
]


def parametri_da_nome(nome):
    """(tensione, ampiezza LED) dal nome del file, come stringhe; ampiezza None se manca.

    Nel nome il punto decimale e' scritto '-' (51-5V = 51.5 V), perche' un
    punto nel nome confonderebbe l'estensione.
    """
    m_v = re.search(r"_(\d+(?:-\d+)?)V_", nome)
    if m_v is None:
        raise ValueError(f"{nome}: tensione non trovata nel nome (atteso es. _51-5V_)")
    m_led = re.search(r"_(\d+(?:-\d+)?)LEDampl_", nome)
    led = m_led.group(1).replace("-", ".") if m_led else None
    return m_v.group(1).replace("-", "."), led


parametri = {nome: parametri_da_nome(nome) for nome in RUN_FILES}
tensioni = [v for v, _ in parametri.values()]
ampiezze = [a for _, a in parametri.values()]

# Lo scan si riconosce da quale parametro distingue tutti i run. Se lo fanno
# entrambi (es. un solo run) vince la tensione, come prima.
if len(set(tensioni)) == len(tensioni):
    SCAN = "HV"
elif None not in ampiezze and len(set(ampiezze)) == len(ampiezze):
    SCAN = "LED"
else:
    SCAN = "HV+LED"

# label -> file, tensione [V], ampiezza LED, descrizione per titoli e stampe
RUNS_G2 = {}
VBIAS = {}
LED_AMPL = {}
DESCRIZIONE = {}
for nome, (v, a) in parametri.items():
    if SCAN == "HV":
        label = v
    elif SCAN == "LED":
        label = a
    else:
        label = f"{v}V_{a}LED" if a is not None else f"{v}V"
    # Due run con stessa tensione e stesso LED (es. ripetuti, _0000 e _0001)
    # avrebbero la stessa label e uno sovrascriverebbe l'altro senza dirlo
    if label in RUNS_G2:
        raise ValueError(f"{nome} e {RUNS_G2[label]}: stessa tensione e stessa ampiezza LED, "
                         f"non si distinguono. Toglierne uno da RUN_FILES.")
    RUNS_G2[label] = nome
    VBIAS[label] = float(v)
    LED_AMPL[label] = float(a) if a is not None else np.nan
    DESCRIZIONE[label] = f"{v} V" + (f", LED {a}" if a is not None else "")

print(f"Scan in {SCAN}: label {list(RUNS_G2)}")

CHANNEL = 16                 # NUMERO di canale V1742, non posizione nel file
MAX_EVENTS = None            # None = tutti


# ============================================================
# LETTURA
# ============================================================

wf_gain2 = {}
SAMPLING_TIME = None
for label, nome in RUNS_G2.items():
    path = os.path.join(DATA_DIR, nome)
    wf, dt_ns = leggi_canale(path, CHANNEL, MAX_EVENTS)
    # Una run fermata prima del primo trigger produce un file valido ma vuoto:
    # senza questo controllo l'errore esce piu' avanti, oscuro (rng.choice su 0 eventi)
    if wf.shape[0] == 0:
        raise ValueError(f"{nome}: 0 eventi nel file (run senza trigger? vedere il .log)")
    if SAMPLING_TIME is not None and not np.isclose(dt_ns, SAMPLING_TIME):
        raise ValueError(f"{nome}: SamplingTime {dt_ns} ns diverso dagli altri run ({SAMPLING_TIME} ns)")
    SAMPLING_TIME = dt_ns    # [ns]
    wf_gain2[label] = wf
    print(f"[{DESCRIZIONE[label]}] {nome}: {wf.shape[0]} eventi x {wf.shape[1]} campioni, dt = {dt_ns:.4f} ns")

N_SAMPLES = next(iter(wf_gain2.values())).shape[1]


def compute_charge(wf, gate_ns, baseline_ns=None):
    """Carica per evento: somma dei campioni ADC dentro il gate [ns].

    Con baseline_ns si sottrae a ogni evento la SUA baseline, media dei
    campioni in quella finestra. Serve: la baseline si muove da un evento
    all'altro (rumore lento, sotto i 50 MHz, che il filtro non toglie), e
    sommata su tanti campioni allarga la distribuzione piu' della distanza
    fra i picchi di singolo fotoelettrone, che spariscono. Misurato sul run
    54 V senza LED: senza sottrazione nessun picco, con la sottrazione e un
    gate stretto attorno all'impulso si separano 1, 2 e 3 p.e.
    Senza baseline_ns la somma e' grezza (baseline compresa).
    I tempi ripartono da 0 sulla matrice ricevuta.
    Gli impulsi di questo SiPM sono positivi: piu' fotoni, carica piu' alta.
    """
    t = np.arange(wf.shape[1]) * SAMPLING_TIME
    sel = (t >= gate_ns[0]) & (t <= gate_ns[1])
    # float64: con le WF filtrate (float) una somma in interi troncherebbe ogni campione
    q = wf[:, sel].sum(axis=1, dtype=np.float64)
    if baseline_ns is not None:
        sel_bl = (t >= baseline_ns[0]) & (t <= baseline_ns[1])
        q -= wf[:, sel_bl].mean(axis=1, dtype=np.float64) * sel.sum()
    return q

In [ ]:
# === Run di cui si mostrano i grafici ===
# Con molti run il notebook arriva a centinaia di grafici e decine di MB: lento da aprire
# e da salvare. Le celle successive calcolano e stampano comunque per TUTTI i
# run; questa scelta salta solo il disegno.
# Per vederli tutti: commentare la seconda riga e rieseguire questa cella
# (la prima serve proprio a questo: senza, commentando la seconda il kernel
# terrebbe il valore vecchio).
RUN_DA_MOSTRARE = None          # None = grafici di tutti i run
RUN_DA_MOSTRARE = ["54"]      # label come in RUNS_G2: tensione ("52") o ampiezza LED ("3")

if RUN_DA_MOSTRARE is not None:
    # Una label sbagliata ("51.50" invece di "51.5") non darebbe nessun grafico senza dirlo
    sconosciute = [l for l in RUN_DA_MOSTRARE if l not in RUNS_G2]
    if sconosciute:
        raise ValueError(f"Run non nella lista: {sconosciute}. Disponibili: {list(RUNS_G2)}")


def mostra(label):
    """True se i grafici di questo run vanno disegnati."""
    return RUN_DA_MOSTRARE is None or label in RUN_DA_MOSTRARE


In [ ]:
# === Alcune waveform di ogni run, per vedere come sono i segnali ===
N_WF_SINGOLE = 6          # waveform mostrate una per pannello
N_WF_SOVRAPPOSTE = 50     # waveform sovrapposte nel pannello finale
EVENTI_SCELTI = None      # lista di indici, es. [0, 10, 250]; None = a caso
SEED = 1                  # cambiarlo per vedere altri eventi a caso

t_wf = np.arange(N_SAMPLES) * SAMPLING_TIME
eventi_per_run = {}       # gli stessi eventi si riusano nella cella del filtro

for lab in wf_gain2:
    wf_run = wf_gain2[lab]
    rng = np.random.default_rng(SEED)
    eventi = (list(EVENTI_SCELTI) if EVENTI_SCELTI is not None
              else sorted(rng.choice(len(wf_run), N_WF_SINGOLE, replace=False)))
    eventi_per_run[lab] = eventi
    if not mostra(lab):
        continue

    ncol = 3
    nrow = int(np.ceil(len(eventi) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), sharex=True, squeeze=False)
    for ax, ev in zip(axes.flat, eventi):
        ax.plot(t_wf, wf_run[ev], lw=0.8, color="k")
        ax.set_title(f"event {ev}")
        ax.set_ylabel("ADC")
        ax.grid(True, alpha=0.3)
    for ax in axes.flat[len(eventi):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("Time [ns]")
    fig.suptitle(f"{DESCRIZIONE[lab]} — single waveforms")
    plt.tight_layout()
    plt.show()

    # Sovrapposte + media su tutti gli eventi: la media mostra dove cade il segnale
    sovr = rng.choice(len(wf_run), min(N_WF_SOVRAPPOSTE, len(wf_run)), replace=False)
    fig, ax = plt.subplots(figsize=(15, 5))
    for ev in sovr:
        ax.plot(t_wf, wf_run[ev], lw=0.5, alpha=0.3)
    ax.plot(t_wf, wf_run.mean(axis=0), lw=2, color="k", label=f"mean over {len(wf_run)} events")
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("ADC")
    ax.set_title(f"{DESCRIZIONE[lab]} — {len(sovr)} overlaid waveforms")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# === Filtro delle waveform: grezza e filtrata a confronto ===
from scipy.signal import butter, iirnotch, sosfiltfilt, filtfilt

# Lo spettro del rumore (run 54 V senza LED) ha una riga dominante a ~98 MHz,
# larga una decina di MHz, e una minore a ~188 MHz; il segnale del SiPM sta
# sotto ~30 MHz (salita ~20 ns, discesa di centinaia di ns). Un passa-basso a
# 50 MHz toglie entrambe le righe senza toccare l'impulso. I notch sono
# l'alternativa se si vuole lasciare intatto il resto della banda: con Q alto
# pero' sono piu' stretti della riga a 98 MHz, che e' larga.
# Si filtra con filtfilt (avanti e indietro): fase nulla, quindi l'impulso non
# si sposta nel tempo e l'inizio della salita resta dov'e'.
# Lo stesso filtro si applica a tutti i run.
FILTRO_LOWPASS = True
LP_CUTOFF_MHZ = 50.0
LP_ORDINE = 4

FILTRO_NOTCH = False
NOTCH_MHZ = [97.5, 187.5]
NOTCH_Q = 10                  # Q = f0 / larghezza: 10 a 98 MHz -> ~10 MHz

USA_FILTRATE = True           # le celle successive analizzano le WF filtrate

fs_hz = 1e9 / SAMPLING_TIME   # SAMPLING_TIME e' in ns


def filtra_waveform(wf):
    """Filtra una matrice (eventi, campioni) lungo i campioni; restituisce float32."""
    out = wf.astype(np.float64)
    if FILTRO_NOTCH:
        for f0 in NOTCH_MHZ:
            b, a = iirnotch(f0 * 1e6, NOTCH_Q, fs_hz)
            out = filtfilt(b, a, out, axis=1)
    if FILTRO_LOWPASS:
        sos = butter(LP_ORDINE, LP_CUTOFF_MHZ * 1e6, btype="low", fs=fs_hz, output="sos")
        out = sosfiltfilt(sos, out, axis=1)
    return out.astype(np.float32)


wf_filt_gain2 = {label: filtra_waveform(wf_gain2[label]) for label in wf_gain2}
wf_analisi = wf_filt_gain2 if USA_FILTRATE else wf_gain2

t_f = np.arange(N_SAMPLES) * SAMPLING_TIME
filtri = " + ".join(([f"low-pass {LP_CUTOFF_MHZ:g} MHz"] if FILTRO_LOWPASS else []) +
                    ([f"notch {NOTCH_MHZ} MHz (Q={NOTCH_Q})"] if FILTRO_NOTCH else [])) or "none"
print(f"Filtro: {filtri}")

for lab in wf_gain2:
    raw, filt = wf_gain2[lab], wf_filt_gain2[lab]
    print(f"[{DESCRIZIONE[lab]}] rumore rms per evento (mediana): grezza {np.median(raw.std(axis=1)):.2f} ADC, "
          f"filtrata {np.median(filt.std(axis=1)):.2f} ADC")

    if not mostra(lab):
        continue

    # --- Stessi eventi della cella delle waveform, grezza e filtrata sovrapposte ---
    eventi = eventi_per_run[lab]
    ncol = 3
    nrow = int(np.ceil(len(eventi) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow), sharex=True, squeeze=False)
    for ax, ev in zip(axes.flat, eventi):
        ax.plot(t_f, raw[ev], lw=0.6, color="gray", alpha=0.7, label="raw")
        ax.plot(t_f, filt[ev], lw=1.2, color="red", label="filtered")
        ax.set_title(f"event {ev}")
        ax.set_ylabel("ADC")
        ax.grid(True, alpha=0.3)
    for ax in axes.flat[len(eventi):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("Time [ns]")
    axes.flat[0].legend(loc="upper right", fontsize=8)
    fig.suptitle(f"{DESCRIZIONE[lab]} — raw vs filtered ({filtri})")
    plt.tight_layout()
    plt.show()

    # --- Waveform media, prima e dopo: mostra se il filtro deforma l'impulso ---
    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.plot(t_f, raw.mean(axis=0), color="gray", label="raw mean")
    ax.plot(t_f, filt.mean(axis=0), color="red", label="filtered mean")
    ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title(f"{DESCRIZIONE[lab]} — raw and filtered mean waveform")
    plt.tight_layout()
    plt.show()

In [ ]:
# === Spettro del rumore: dove sono le bande, prima e dopo il filtro ===
# Come si legge: la FFT scompone ogni waveform in sinusoidi; la potenza media
# su molti eventi dice quanta ampiezza c'e' a ogni frequenza. Il segnale del
# SiPM e' lento e sta tutto in basso (sotto ~30 MHz). Un rumore periodico
# compare invece come un picco stretto che spicca sul fondo: e' una "riga".
# Le righe si trovano da sole (find_peaks sul logaritmo della potenza grezza)
# e si marcano in tutti e due i pannelli, cosi' si vede quanto le ha tolte il filtro.
from scipy.signal import find_peaks, peak_widths

N_EVENTI_SPETTRO = 5000      # bastano per una media stabile
PROMINENZA_DECADI = 1.0      # una riga deve spiccare di 10x sul fondo vicino; con 3x
                             # si prendono anche le fluttuazioni del fondo
F_MIN_RIGHE_MHZ = 35.0       # sotto c'e' il segnale: non lo si cerca come rumore
BANDA_SEGNALE_MHZ = 30.0


def spettro_medio(wf):
    """(frequenze [MHz], potenza media [ADC^2]) su N_EVENTI_SPETTRO eventi.

    Si toglie la media di ogni evento (se no domina la frequenza zero) e si
    applica una finestra di Hann, che evita che i bordi della traccia, tagliati
    di netto, spargano potenza su tutte le frequenze.
    """
    x = wf[:N_EVENTI_SPETTRO].astype(np.float64)
    finestra = np.hanning(x.shape[1])
    x = (x - x.mean(axis=1, keepdims=True)) * finestra
    pot = (np.abs(np.fft.rfft(x, axis=1)) ** 2).mean(axis=0) / (finestra ** 2).sum()
    freq = np.fft.rfftfreq(x.shape[1], d=SAMPLING_TIME * 1e-9) / 1e6
    return freq, pot


for lab in wf_gain2:
    freq, pot_raw = spettro_medio(wf_gain2[lab])
    _, pot_filt = spettro_medio(wf_filt_gain2[lab])
    df_mhz = freq[1] - freq[0]

    # Righe di rumore sullo spettro grezzo
    cerca = freq >= F_MIN_RIGHE_MHZ
    idx, prop = find_peaks(np.log10(pot_raw[cerca]), prominence=PROMINENZA_DECADI)
    idx = np.flatnonzero(cerca)[idx]
    # larghezza a meta' altezza in scala logaritmica, convertita in MHz
    larg = peak_widths(np.log10(pot_raw), idx, rel_height=0.5)[0] * df_mhz

    print(f"=== {DESCRIZIONE[lab]} ===")
    print(f"Risoluzione in frequenza: {df_mhz:.3f} MHz  (Nyquist {freq[-1]:.0f} MHz)")
    print(f"{'riga [MHz]':>11} {'larghezza [MHz]':>16} {'sopra il fondo':>15} {'attenuazione filtro':>20}")
    for i, w, prom in zip(idx, larg, prop["prominences"]):
        att = pot_raw[i] / pot_filt[i]
        # oltre ~1e6 il filtrato e' sul pavimento numerico (float32): il numero non e' una misura
        att_txt = f"{att:19.1e}x" if att < 1e6 else f"{'> 1e6x':>20}"
        print(f"{freq[i]:11.1f} {w:16.1f} {10 ** prom:14.0f}x {att_txt}")

    if not mostra(lab):
        continue
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
    for ax, pot, titolo, colore in [(axes[0], pot_raw, "BEFORE the filter", "gray"),
                                    (axes[1], pot_filt, f"AFTER the filter ({filtri})", "red")]:
        if pot is pot_filt:
            ax.semilogy(freq, pot_raw, color="gray", lw=0.8, alpha=0.35, label="before (reference)")
        ax.semilogy(freq, pot, color=colore, lw=1, label="spectrum")
        ax.axvspan(0, BANDA_SEGNALE_MHZ, color="green", alpha=0.10, label=f"signal band (< {BANDA_SEGNALE_MHZ:g} MHz)")
        for k, (i, w) in enumerate(zip(idx, larg)):
            ax.axvspan(freq[i] - w / 2, freq[i] + w / 2, color="orange", alpha=0.35,
                       label="noise lines" if k == 0 else None)
            ax.annotate(f"{freq[i]:.1f} MHz", (freq[i], pot_raw[i]), textcoords="offset points",
                        xytext=(4, 4), fontsize=8)
        if FILTRO_LOWPASS:
            ax.axvline(LP_CUTOFF_MHZ, color="blue", ls="--", label=f"low-pass cutoff {LP_CUTOFF_MHZ:g} MHz")
        if FILTRO_NOTCH:
            for k, f0 in enumerate(NOTCH_MHZ):
                ax.axvline(f0, color="purple", ls=":", label="notch" if k == 0 else None)
        ax.set_title(titolo)
        ax.set_xlabel("Frequency [MHz]")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8, loc="upper right")
    axes[0].set_ylabel("Mean power [ADC$^2$]")
    # Sotto ~1e-4 del fondo grezzo il filtrato tocca la precisione dei float32:
    # mostrarlo schiaccerebbe tutto il resto senza aggiungere informazione
    axes[0].set_ylim(np.median(pot_raw) * 1e-4, pot_raw.max() * 10)
    fig.suptitle(f"{DESCRIZIONE[lab]} — mean spectrum over {min(N_EVENTI_SPETTRO, len(wf_gain2[lab]))} events")
    plt.tight_layout()
    plt.show()

In [ ]:
t = np.arange(N_SAMPLES) * SAMPLING_TIME

# === Gate e baseline: nella stessa posizione rispetto all'impulso che nel notebook Gain ===
# Il Q_pe del gate stretto e' la carica di un p.e. DENTRO IL GATE: si
# confronta con quello del LED solo se il gate copre la stessa parte
# dell'impulso. In tempo assoluto i due notebook non si somigliano (qui il
# trigger mette l'impulso a ~452 ns, col LED a ~204), quindi gate e baseline
# si mettono alla stessa DISTANZA dall'inizio della salita, in campioni.
# Inizio salita mediano (5 sigma, cella sotto), misurato da 55 a 57 V, dove
# lo si trova in quasi tutti gli eventi; sotto i 54 V l'impulso e' troppo
# vicino al rumore e la mediana prende fluttuazioni a caso:
#   LED  : 204-205 ns -> gate (220, 270) = 12 campioni dopo, 38 campioni;
#          baseline (0, 180) = finisce 18 campioni prima della salita, 136 campioni
#   buio : 452-453 ns -> gate (468, 518), baseline (248, 428): stessi numeri.
# Il gate precedente (470, 520) cominciava 13 campioni dopo la salita invece di 12.
# La baseline non parte da 0 come col LED: nei primi ~150 ns c'e' una struttura
# legata all'inizio della traccia (2.7 ADC a 0 ns, -0.7 a 100), uguale nelle due
# modalita'; nella baseline sposta tutte le cariche della stessa quantita',
# quindi il piedistallo ma non la distanza fra i picchi.
GATE_NS = (468, 518)
BASELINE_NS = (248, 428)      # prima della salita, senza segnale

# === Eventi con un impulso nella baseline ===
# Al buio il SiPM ha impulsi a caso a un rate di MHz: uno che cade nella
# baseline alza la baseline dell'evento, e la carica nel gate esce troppo bassa
# o negativa. Misurato a 54 V con la baseline (200, 400): fra gli eventi con
# un impulso nella baseline (massimo sopra 4 rms) il 32% ha carica negativa,
# fra quelli puliti il 4%. Nel fit di Borel non c'e' nessun termine per questi
# eventi, che si mescolerebbero al piedistallo.
# Il taglio non distorce Q_pe ne' lambda: un impulso a caso nella baseline e'
# indipendente dall'impulso che ha fatto scattare il trigger.
# La frazione di eventi tagliati dipende dal DCR (e servira' a misurarlo).
# Il rumore e' l'rms robusto (MAD) della baseline, mediana sugli eventi: con
# la deviazione standard gli impulsi stessi lo gonfierebbero.
TAGLIO_BASELINE = True
N_RMS_BASELINE = 4

# Si parte sempre da tutti gli eventi: rieseguendo la cella non si taglia due volte
wf_tutte = wf_filt_gain2 if USA_FILTRATE else wf_gain2
sel_bl = (t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])
wf_analisi = {}
pulite_gain2 = {}             # label -> maschera degli eventi tenuti, sugli eventi del file
for lab in wf_tutte:
    w_bl = wf_tutte[lab][:, sel_bl].astype(np.float64)
    w_bl -= w_bl.mean(axis=1, keepdims=True)
    rms = 1.4826 * np.median(np.median(np.abs(w_bl), axis=1))
    pulita = w_bl.max(axis=1) < N_RMS_BASELINE * rms if TAGLIO_BASELINE else np.ones(len(w_bl), bool)
    pulite_gain2[lab] = pulita
    wf_analisi[lab] = wf_tutte[lab][pulita]
    print(f"[{DESCRIZIONE[lab]}] rumore baseline {rms:.2f} ADC; eventi con un impulso nella baseline "
          f"(> {N_RMS_BASELINE} rms): {np.count_nonzero(~pulita)} su {len(pulita)} "
          f"({100 * np.mean(~pulita):.1f}%)" + ("" if TAGLIO_BASELINE else " -- NON tagliati"))

# === Inizio della salita del segnale ===
# Inizio = primo campione che supera la baseline di N_SIGMA_START volte il suo
# rumore. Si misura sia sulla waveform media sia evento per evento: la media
# dice dov'e' il segnale, la distribuzione per evento quanto si sposta (jitter),
# che e' quello che conta per scegliere il gate.
BASELINE_NS_START = (0, 150)   # regione senza segnale, prima della salita
N_SIGMA_START = 5

bl_sel = (t >= BASELINE_NS_START[0]) & (t <= BASELINE_NS_START[1])
dopo_bl = t > BASELINE_NS_START[1]
wf_media_gain2 = {}            # waveform media di ogni run, riusata qui sotto
salita_gain2 = {}              # inizio salita di ogni run

for lab in wf_analisi:
    wf_s = wf_analisi[lab].astype(np.float64)

    # Waveform media
    wf_media = wf_s.mean(axis=0)
    wf_media_gain2[lab] = wf_media
    bl_media, rms_media = wf_media[bl_sel].mean(), wf_media[bl_sel].std()
    sopra = (wf_media > bl_media + N_SIGMA_START * rms_media) & dopo_bl
    t_start_media = t[np.argmax(sopra)] if sopra.any() else np.nan
    # Tempo al 10% dell'ampiezza: meno sensibile al rumore della soglia in sigma
    picco_media = wf_media.max()
    sopra10 = (wf_media > bl_media + 0.1 * (picco_media - bl_media)) & dopo_bl
    t10_media = t[np.argmax(sopra10)]

    # Evento per evento, ciascuno con la sua baseline e il suo rumore
    bl_ev = wf_s[:, bl_sel].mean(axis=1, keepdims=True)
    rms_ev = wf_s[:, bl_sel].std(axis=1, keepdims=True)
    sopra_ev = (wf_s > bl_ev + N_SIGMA_START * rms_ev) & dopo_bl
    trovato = sopra_ev.any(axis=1)
    t_start_ev = np.where(trovato, t[np.argmax(sopra_ev, axis=1)], np.nan)
    salita_gain2[lab] = {"t_start_media": t_start_media, "t10_media": t10_media, "t_start_ev": t_start_ev}

    print(f"=== {DESCRIZIONE[lab]} ===")
    print(f"Waveform media: baseline {bl_media:.1f} ADC, rumore {rms_media:.2f} ADC")
    print(f"  inizio salita ({N_SIGMA_START} sigma) = {t_start_media:.1f} ns")
    print(f"  10% dell'ampiezza             = {t10_media:.1f} ns")
    print(f"Per evento ({trovato.sum()}/{len(trovato)} con salita trovata):")
    print(f"  mediana {np.nanmedian(t_start_ev):.1f} ns, "
          f"16-84% [{np.nanpercentile(t_start_ev, 16):.1f}, {np.nanpercentile(t_start_ev, 84):.1f}] ns, "
          f"min {np.nanmin(t_start_ev):.1f}  max {np.nanmax(t_start_ev):.1f} ns")
    # il controllo del gate: col LED il gate parte 12 campioni dopo la salita mediana
    print(f"  gate dopo la salita mediana: {(GATE_NS[0] - np.nanmedian(t_start_ev)) / SAMPLING_TIME:.0f} campioni "
          f"(LED: 12; non vale sotto i 54 V, salita non trovata)")

    if not mostra(lab):
        continue
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
    zoom = (t >= t_start_media - 100) & (t <= t_start_media + 150)
    ax1.plot(t[zoom], wf_media[zoom], color="k", label="mean waveform")
    ax1.axhline(bl_media + N_SIGMA_START * rms_media, color="gray", ls=":", label=f"baseline + {N_SIGMA_START} sigma")
    ax1.axvline(t_start_media, color="red", label=f"rise start {t_start_media:.1f} ns")
    ax1.axvline(t10_media, color="orange", ls="--", label=f"10% amplitude {t10_media:.1f} ns")
    ax1.set_xlabel("Time [ns]"); ax1.set_ylabel("ADC"); ax1.grid(True, alpha=0.3); ax1.legend()
    ax2.hist(t_start_ev[trovato], bins=np.arange(np.nanmin(t_start_ev), np.nanmax(t_start_ev) + 2 * SAMPLING_TIME, SAMPLING_TIME) - SAMPLING_TIME / 2, color="steelblue")
    ax2.set_xlabel("Rise start per event [ns]"); ax2.set_ylabel("Events"); ax2.set_yscale("log"); ax2.grid(True, alpha=0.3)
    fig.suptitle(f"{DESCRIZIONE[lab]} — rise start")
    plt.tight_layout()
    plt.show()

# === Carica per evento ===
# Gate stretto attorno all'impulso e baseline per evento subito prima: sono
# le due scelte che fanno comparire i picchi di singolo fotoelettrone (vedi
# compute_charge). Integrare tutta la traccia somma solo rumore in piu'.
charge_gain2 = {label: compute_charge(wf_analisi[label], GATE_NS, BASELINE_NS) for label in wf_analisi}

for lab in wf_analisi:
    if not mostra(lab):
        continue
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(t, wf_media_gain2[lab], color="k", label="mean waveform")
    ax.axvspan(*BASELINE_NS, color="green", alpha=0.15, label=f"baseline {BASELINE_NS} ns")
    ax.axvspan(*GATE_NS, color="orange", alpha=0.3, label=f"gate {GATE_NS} ns")
    ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title(f"{DESCRIZIONE[lab]} — gate and baseline on the mean waveform")
    plt.tight_layout()
    plt.show()


In [ ]:
# === Distribuzione di carica di ogni run + modello per il fit di Borel ===
from math import factorial

# --- Intervallo e binnaggio ---
# Come nel notebook Gain: tutto l'intervallo dei dati e Freedman-Diaconis.
# L'istogramma si fa solo in CHARGE_RANGE: a parita' di numero di bin,
# un intervallo piu' stretto da' bin piu' fini. Gli eventi fuori restano
# in charge_gain2, ma non entrano ne' nell'istogramma ne' nel fit.
#   None            -> tutto l'intervallo dei dati di ogni run (come nel Gain)
#   "auto"          -> per run, fra i percentili CHARGE_PERCENTILI
#   (lo, hi)        -> stesso intervallo [ADC] per tutti i run
#   {"54": (lo, hi)} -> per run a mano; i run assenti vanno in automatico
CHARGE_RANGE = None
CHARGE_PERCENTILI = (0.1, 99.5) # [%], solo con "auto"
N_BINS = None                   # None = regola di Freedman-Diaconis; o un numero fisso
LOG_Y = False

# Istogramma di ogni run: label -> counts, centers, bin_edges, bin_width
# (servono al fit e alla cella delle waveform per picco)
hist_gain2 = {}


# --- Modello: perche' Borel e non la Poisson generalizzata del Gain ---
# Col LED il trigger e' il lampo: il numero di fotoelettroni primari e'
# poissoniano, 0 compreso, e ognuno si porta dietro il suo crosstalk
# (Poisson generalizzata, parametri mu e lambda).
# Qui il trigger e' l'impulso stesso: ogni evento ha UN fotoelettrone primario
# (un impulso al buio) e i picchi oltre il primo vengono solo dal crosstalk.
# Il numero di celle accese segue allora la distribuzione di Borel con lo
# stesso lambda: P(n) = exp(-n*lambda) * (n*lambda)^(n-1) / n!, n >= 1, media
# 1/(1-lambda). E' la Poisson generalizzata con un solo primario certo:
# lambda ha lo stesso significato nei due notebook e si possono confrontare.
# Il picco a 0 p.e. qui non ha la Poisson dietro: sono gli eventi in cui il gate
# non contiene l'impulso del trigger (trigger sul rumore, impulso fuori dal
# gate per il jitter). Ha un peso libero, f0.
#
# --- La Borel pura non basta ---
# A 54 V dava lambda = 0.25 con chi2/ndf 28, contro 0.141 del LED: il picco a
# 2 p.e. era il 33% di quello a 1 p.e. e il 3 il 17% del 2, due rapporti che
# nessuna Borel produce insieme (il secondo dovrebbe essere il piu' grande).
# Dal 2 p.e. in su la forma e' invece una Borel: manca una parte del picco a
# 1 p.e. Tre modelli (MODELLO_BUIO), misurati sullo scan 53.5-56.5 V del 2026-10-06:
#
#  "eps1": il picco a 1 p.e. scalato di un fattore libero eps1, il resto Borel.
#     lambda coincide con quello del LED entro il 2% da 53.5 a 55.5 V
#     (0.101/0.100, 0.141/0.141, 0.186/0.192, 0.237/0.240, 0.284/0.290),
#     a 56 e 56.5 V resta sotto del 10% (picchi sovrapposti, fit peggiore).
#     eps1 vale ~0.37 fino a 55 V e poi cala: 0.31, 0.21, 0.09.
#     PERCHE' manchino quegli impulsi NON e' capito. Non e' la soglia del
#     self-trigger: nei log la soglia assoluta e' ferma a 982-983 conteggi su
#     tutto lo scan, e con una soglia fissa eps1 dovrebbe CRESCERE con la
#     tensione (l'impulso da 1 p.e. cresce), non calare.
#  "pileup": oltre all'impulso del trigger, altri impulsi al buio a caso nel
#     gate (Poisson generalizzata, media mu_dc, stesso lambda). Escluso: lambda
#     esce 0.015-0.14 contro 0.10-0.29 del LED, con chi2 uguale a "eps1". Gli
#     impulsi a caso spostano in su tutti i picchi, i dati ne svuotano uno solo.
#  "borel": la Borel pura.
# Con "eps1" lambda e' confrontabile col LED; eps1 resta un parametro descrittivo.
MODELLO_BUIO = "eps1"


def borel_pmf(n, lam):
    """Borel: probabilita' di n celle accese partendo da un primario, n >= 1."""
    if n < 1:
        return 0.0
    return np.exp(-n * lam) * (n * lam) ** (n - 1) / factorial(n)


def gen_poisson_pmf(n, mu, lam):
    """Generalized Poisson (Borel-Poisson): P(n; mu, lambda). Uguale a quella del notebook Gain."""
    if n == 0:
        return np.exp(-mu)
    return mu * (mu + n * lam) ** (n - 1) * np.exp(-(mu + n * lam)) / factorial(n)


def peso_buio(n, f0, lam, par3=None):
    """Peso del picco a n p.e. secondo MODELLO_BUIO; par3 e' mu_dc ("pileup") o eps1 ("eps1").

    f0 va al picco a 0 p.e., il resto (1 - f0) ai picchi da 1 in su.
    """
    if n == 0:
        return f0
    if MODELLO_BUIO == "pileup":
        # n celle = k dall'impulso del trigger (k >= 1) + n - k dagli impulsi a caso
        return (1 - f0) * sum(borel_pmf(k, lam) * gen_poisson_pmf(n - k, par3, lam) for k in range(1, n + 1))
    if MODELLO_BUIO == "eps1":
        # rinormalizzato a 1 - f0: eps1 dice quanti impulsi da 1 p.e. mancano, non cambia il totale
        b1 = borel_pmf(1, lam)
        return (1 - f0) * borel_pmf(n, lam) * (par3 if n == 1 else 1.0) / (par3 * b1 + (1 - b1))
    return (1 - f0) * borel_pmf(n, lam)


for lab in charge_gain2:
    charge = charge_gain2[lab]
    n_eventi = len(charge)

    if CHARGE_RANGE is None:
        lo, hi = charge.min(), charge.max()
    elif CHARGE_RANGE == "auto" or (isinstance(CHARGE_RANGE, dict) and lab not in CHARGE_RANGE):
        lo, hi = np.percentile(charge, CHARGE_PERCENTILI)
    elif isinstance(CHARGE_RANGE, dict):
        lo, hi = CHARGE_RANGE[lab]
    else:
        lo, hi = CHARGE_RANGE
    in_range = (charge >= lo) & (charge <= hi)
    charge_sel = charge[in_range]
    n_sel = len(charge_sel)
    # Se nessun evento cade nell'intervallo, np.percentile fallisce con un IndexError
    # che non dice nulla: succede quando CHARGE_RANGE e' stato scelto per un altro run
    if n_sel == 0:
        p1, p99 = np.percentile(charge, [1, 99])
        raise ValueError(f"{DESCRIZIONE[lab]}: nessun evento in CHARGE_RANGE = ({lo}, {hi}) ADC; "
                         f"le cariche stanno fra {p1:.0f} e {p99:.0f} ADC (1-99%)")

    # Freedman-Diaconis, larghezza = 2*IQR / N^(1/3), calcolata sugli eventi
    # nell'intervallo: scala con quanti eventi finiscono davvero nell'istogramma
    # e usa l'IQR invece della deviazione standard, quindi non si allarga per le code.
    if N_BINS is None:
        q25, q75 = np.percentile(charge_sel, [25, 75])
        n_bins = max(1, int(np.ceil((hi - lo) / (2 * (q75 - q25) / n_sel ** (1 / 3)))))
    else:
        n_bins = int(N_BINS)
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    counts, _ = np.histogram(charge_sel, bins=bin_edges)
    hist_gain2[lab] = {"counts": counts, "centers": centers, "bin_edges": bin_edges, "bin_width": bin_width}

    print(f"[{DESCRIZIONE[lab]}] {n_eventi} eventi")
    print(f"  carica: min {charge.min():.0f}  max {charge.max():.0f}  mediana {np.median(charge):.0f} ADC")
    print(f"  intervallo [{lo:.0f}, {hi:.0f}] ADC: {n_sel} eventi dentro, {n_eventi - n_sel} fuori")
    print(f"  binnaggio: {n_bins} bin da {bin_width:.1f} ADC")

    if not mostra(lab):
        continue
    # --- Plot della distribuzione ---
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.hist(charge_sel, bins=bin_edges, histtype="step", color="k", label=f"{DESCRIZIONE[lab]}  ({n_sel} events)")
    ax.set_xlim(lo, hi)
    ax.set_xlabel("Charge [ADC]")
    ax.set_ylabel(f"Events / {bin_width:.1f} ADC")
    ax.set_title(f"Charge distribution — {DESCRIZIONE[lab]}")
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# === Fit di Borel: Q_pe dalle distanze fra i picchi e crosstalk lambda ===
# Stesso fit del notebook Gain (posizioni dei picchi libere, gaussiane con
# sigma_n = sqrt(sigma0^2 + n*sigma1^2), errori poissoniani, chi2 ridotto),
# con i pesi di Borel al posto della Poisson generalizzata (cella sopra).
# Le differenze, tutte dovute al trigger:
#  - i picchi liberi partono da 1 p.e.: il piedistallo qui ha pochi eventi e
#    la sua posizione non si fissa; sta a un passo sotto il picco a 1 p.e.;
#  - Q_pe e' la media delle distanze fra i picchi liberi da 1 p.e. in su,
#    cioe' (ultimo - primo) / (K - 1); nel Gain il primo e' il piedistallo;
#  - i parametri dei pesi sono f0, lambda e quello del MODELLO_BUIO (cella sopra).
# I picchi oltre l'ultimo libero (la coda, pesata dalla Borel) stanno a passo
# costante, pari alla media delle distanze, come nel Gain.
#
from scipy.signal import find_peaks

N_LIBERI_MAX = 8          # picchi con posizione libera, al massimo
P_CODA = 1e-4             # si aggiungono picchi finche' la probabilita' oltre e' sotto questa
# Dove i picchi non si separano la ricerca automatica non trova il passo: lo
# si da' a mano, con quanti picchi lasciare liberi (da 1 p.e. in su).
# label -> (passo iniziale [ADC], picchi liberi). Il picco a 1 p.e. parte da un passo.
SEMI_MANUALI = {}
FORZA_AFFIDABILE = set()  # run da tenere anche se i picchi non risultano separati
FRAZ_PICCO_DISEGNATO = 0.01   # picchi singoli disegnati se alti almeno questa frazione del piu' alto

fit_gain2 = {}            # label -> risultati del fit, per le celle successive


def passo_autocorr(counts, bin_width):
    """Passo fra i picchi dall'autocorrelazione dell'istogramma [ADC], o nan.

    Con bin larghi quanto un quinto del passo (alle tensioni alte) find_peaks
    sui conteggi prende le fluttuazioni; l'autocorrelazione usa tutti i picchi
    insieme. Si toglie prima l'andamento lento, se no domina il lag zero.
    """
    k = max(3, len(counts) // 8) | 1
    struttura = counts - np.convolve(counts, np.ones(k) / k, mode="same")
    ac = np.correlate(struttura, struttura, mode="full")[len(counts) - 1:]
    ip, _ = find_peaks(ac)
    return ip[0] * bin_width if len(ip) else np.nan


def semina_picchi(counts, centers, passo):
    """Posizioni iniziali dei picchi da 1 p.e. in su: massimo locale attorno a ogni posizione attesa.

    Come nel Gain, ma si parte dal picco a 1 p.e., atteso a un passo dallo
    zero. Si va avanti finche' il massimo spicca sulle valli a meta' passo;
    dal 2 p.e. in poi, se non spicca, i picchi successivi non si vedono piu'.
    """
    liscio = np.convolve(counts, np.ones(3) / 3, mode="same")
    x0, picchi = 0.0, []
    for n in range(1, N_LIBERI_MAX + 1):
        vicino = np.abs(centers - (x0 + n * passo)) < passo / 3
        if not vicino.any():
            break
        i = np.flatnonzero(vicino)[np.argmax(liscio[vicino])]
        valli = [liscio[np.argmin(np.abs(centers - (x0 + (n + s) * passo)))] for s in (-0.5, 0.5)]
        if n >= 2 and liscio[i] < 1.03 * max(valli):
            break
        picchi.append(centers[i])
        x0 = centers[i] - n * passo    # la sequenza segue i picchi trovati
    return np.array(picchi)


def posizioni_picchi(m_liberi, n_tot):
    """Posizioni di tutti i picchi da 0 a n_tot-1 p.e.: piedistallo, liberi, coda a passo medio.

    m_liberi sono i picchi da 1 p.e. in su; il piedistallo sta un passo sotto il primo.
    """
    passo = (m_liberi[-1] - m_liberi[0]) / (len(m_liberi) - 1)
    coda = m_liberi[-1] + passo * np.arange(1, n_tot - len(m_liberi))
    return np.concatenate([[m_liberi[0] - passo], m_liberi, coda])


def picco_singolo(x, n, pesi, sigma0, sigma1, posizione):
    """Densita' del solo picco a n p.e.: gaussiana pesata con peso_buio; pesi = (f0, lambda[, par3])."""
    s = np.sqrt(sigma0 ** 2 + n * sigma1 ** 2)
    return peso_buio(n, *pesi) * np.exp(-(x - posizione) ** 2 / (2 * s ** 2)) / (s * np.sqrt(2 * np.pi))


def posa_etichette(ax, etichette, centers, counts, salti_max=8):
    """Scrive le etichette dei picchi senza sovrapporle fra loro, ai dati o alla legenda.

    Ogni etichetta e' solo il numero di p.e. (la legenda dice cosa sono), e
    parte subito sopra il suo picco. Con la scritta intera "n p.e." a 56 V
    un'etichetta era larga 1600 ADC contro 650 fra due picchi: per non
    sovrapporsi salivano di riga e restavano sospese a meta' fra due o tre
    picchi, centrate giuste ma impossibili da attribuire. Il numero da solo
    e' largo meno di un passo. Se tocca comunque un'altra etichetta, o se sotto
    di lei passa l'istogramma, sale di una riga alla volta e una linea del suo
    colore la collega al picco: chi legge non deve indovinare a quale vada.
    Dopo salti_max righe, o se esce dal grafico, si rinuncia.
    Le sovrapposizioni fra testi si misurano in pixel, a grafico gia'
    impaginato: in coordinate dei dati dipenderebbero dalla scala.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    riquadro = ax.get_window_extent(renderer)
    in_dati = ax.transData.inverted()
    occupati = [ax.get_legend().get_window_extent(renderer)] if ax.get_legend() else []
    for testo, x, y, col in etichette:
        for salto in range(salti_max + 1):
            linea = dict(arrowstyle="-", color=col, lw=0.8, shrinkA=0, shrinkB=1) if salto else None
            a = ax.annotate(testo, (x, y), textcoords="offset points", xytext=(0, 3 + 14 * salto),
                            ha="center", va="bottom", fontsize=12, fontweight="bold", color=col,
                            arrowprops=linea,
                            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.8))
            bb = a.get_window_extent(renderer).padded(2)
            (xd0, yd0), (xd1, _) = in_dati.transform([(bb.x0, bb.y0), (bb.x1, bb.y1)])
            sotto = (centers >= xd0) & (centers <= xd1)
            libero_dai_dati = not sotto.any() or counts[sotto].max() < yd0
            dentro = riquadro.contains(bb.x0, bb.y0) and riquadro.contains(bb.x1, bb.y1)
            if dentro and libero_dai_dati and not any(bb.overlaps(o) for o in occupati):
                occupati.append(bb)
                break
            a.remove()


def densita_picchi_liberi(x, pesi, sigma0, sigma1, m_liberi, n_tot):
    """Somma dei picchi da 0 a n_tot-1 p.e., con le posizioni dei liberi date una per una."""
    m = posizioni_picchi(np.array(m_liberi), n_tot)
    return sum(picco_singolo(x, n, pesi, sigma0, sigma1, m[n]) for n in range(n_tot))


# Parametro in piu' di ciascun modello: nome, valore iniziale, limiti
PAR3 = {"pileup": ("mu_dc", 0.1, (0.0, 3.0)), "eps1": ("eps1", 0.7, (0.01, 1.0)), "borel": None}[MODELLO_BUIO]


for lab in hist_gain2:
    h = hist_gain2[lab]
    counts, centers, w = h["counts"], h["centers"], h["bin_width"]

    if lab in SEMI_MANUALI:
        passo0, k_lib = SEMI_MANUALI[lab]
        m0 = passo0 * np.arange(1, k_lib + 1, dtype=float)
    else:
        passo0 = passo_autocorr(counts, w)
        m0 = semina_picchi(counts, centers, passo0) if np.isfinite(passo0) else np.array([])
    K = len(m0)
    if K < 2:
        print(f"[{DESCRIZIONE[lab]}] meno di 2 picchi trovati da 1 p.e. in su: niente fit. "
              f"Dare il passo in SEMI_MANUALI.")
        continue

    # lambda iniziale dalla carica media: media della Borel = 1/(1-lambda)
    carica_media = np.sum(counts * centers) / counts.sum()
    lam0 = float(np.clip(1 - passo0 / max(carica_media, passo0), 0.02, 0.6))
    n_tot = K + 1
    while n_tot < 60 and 1 - sum(borel_pmf(n, lam0) for n in range(1, n_tot)) > P_CODA:
        n_tot += 1

    norm = counts.sum() * w
    dens = counts / norm
    err = np.sqrt(np.maximum(counts, 1)) / norm
    # Parametri del fit: f0, lambda, [par3], sigma0, sigma1, posizioni dei picchi liberi
    n_pesi = 2 if PAR3 is None else 3
    f = lambda x, *p: densita_picchi_liberi(x, p[:n_pesi], p[n_pesi], p[n_pesi + 1], np.array(p[n_pesi + 2:]), n_tot)
    p_pesi, lo_pesi, hi_pesi = [0.03, lam0], [0, 0], [0.9, 0.9]
    if PAR3 is not None:
        p_pesi.append(PAR3[1]); lo_pesi.append(PAR3[2][0]); hi_pesi.append(PAR3[2][1])
    p0 = [*p_pesi, passo0 / 4, passo0 / 8, *m0]
    lim = ([*lo_pesi, 1, 0, *(m0 - passo0 / 2)], [*hi_pesi, 2 * passo0, 2 * passo0, *(m0 + passo0 / 2)])
    try:
        popt, pcov = curve_fit(f, centers, dens, p0=p0, sigma=err, bounds=lim, maxfev=50000)
    except RuntimeError as e:
        print(f"[{DESCRIZIONE[lab]}] fit non convergente: {e}")
        continue

    chi2_rid = np.sum(((dens - f(centers, *popt)) / err) ** 2) / (len(centers) - len(popt))
    # chi2 ridotto > 1 (il modello non descrive tutto, es. afterpulse fra i picchi):
    # gli errori di curve_fit sono sottostimati e si allargano di sqrt(chi2_rid)
    scala = np.sqrt(max(chi2_rid, 1.0))
    n_forma = n_pesi + 2                    # parametri prima delle posizioni
    errori = np.sqrt(np.diag(pcov)) * scala
    pesi = tuple(popt[:n_pesi])
    f0, lam = pesi[:2]
    e_f0, e_lam = errori[:2]
    par3, e_par3 = (popt[2], errori[2]) if PAR3 is not None else (np.nan, np.nan)
    sigma0, sigma1 = popt[n_pesi:n_forma]
    m = popt[n_forma:]
    distanze = np.diff(m)
    q_pe = distanze.mean()
    a = np.zeros(len(popt)); a[n_forma], a[n_forma + K - 1] = -1 / (K - 1), 1 / (K - 1)
    e_qpe = np.sqrt(a @ pcov @ a) * scala
    # Picchi separati se il picco a 1 p.e. e' piu' stretto di meta' passo. Nel
    # Gain il criterio e' su sigma0, la larghezza del piedistallo: qui il
    # piedistallo quasi non c'e', e il primo picco che conta e' quello a 1 p.e.
    sigma1pe = np.hypot(sigma0, sigma1)
    affidabile = sigma1pe < q_pe / 2 or lab in FORZA_AFFIDABILE
    m_tutti = posizioni_picchi(m, n_tot)

    fit_gain2[lab] = {"q_pe": q_pe, "e_q_pe": e_qpe, "f0": f0, "e_f0": e_f0, "lam": lam, "e_lam": e_lam, "par3": par3, "e_par3": e_par3, "pesi": pesi,
                      "celle_medie": 1 / (1 - lam), "sigma0": sigma0, "sigma1": sigma1,
                      "posizioni": m, "distanze": distanze, "chi2_rid": chi2_rid,
                      "n_tot": n_tot, "affidabile": affidabile, "popt": popt}

    print(f"[{DESCRIZIONE[lab]}] {K} picchi liberi (da 1 p.e.) + coda fino a {n_tot - 1} p.e.  chi2/ndf = {chi2_rid:.2f}"
          + ("" if affidabile else "   NON AFFIDABILE: picchi non separati")
          + ("   (picchi non separati, tenuto per FORZA_AFFIDABILE)" if lab in FORZA_AFFIDABILE else ""))
    print(f"  Q_pe = {q_pe:.1f} ± {e_qpe:.1f} ADC   distanze: {np.round(distanze, 1)}")
    print(f"  lambda = {lam:.4f} ± {e_lam:.4f}   celle per impulso 1/(1-lambda) = {1 / (1 - lam):.3f}   "
          f"P(>=2 p.e.) = {1 - borel_pmf(1, lam):.3f}")
    print(f"  modello {MODELLO_BUIO}" + ("" if PAR3 is None else f": {PAR3[0]} = {par3:.3f} ± {e_par3:.3f}")
          + f"   f0 = {f0:.3f} ± {e_f0:.3f}")
    print(f"  sigma0 = {sigma0:.1f}  sigma1 = {sigma1:.1f} ADC")

    if not mostra(lab):
        continue
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.step(centers, counts, where="mid", color="tab:blue", label=f"{DESCRIZIONE[lab]}")
    xx = np.linspace(centers[0], centers[-1], 2000)
    ax.plot(xx, f(xx, *popt) * norm, color="red",
            label=f"{MODELLO_BUIO}: $Q_{{pe}}$ = {q_pe:.1f} ± {e_qpe:.1f} ADC, $\\lambda$ = {lam:.3f}" + ("" if PAR3 is None else f", {PAR3[0]} = {par3:.2f}"))

    # Ogni picco da solo, tratteggiato: peso, posizione e larghezza sono quelli del
    # fit, niente di rifittato. Solo quelli abbastanza alti: la coda ne ha
    # decine invisibili, che affollerebbero il grafico.
    singoli = [picco_singolo(xx, n, pesi, sigma0, sigma1, m_tutti[n]) * norm for n in range(n_tot)]
    soglia = FRAZ_PICCO_DISEGNATO * max(y.max() for y in singoli)
    etichette = []
    for n, y_n in enumerate(singoli):
        if y_n.max() < soglia:
            continue
        # tab10 senza blu, rosso e grigio: sono gia' istogramma, fit e legenda
        col = plt.cm.tab10([1, 2, 4, 5, 6, 8, 9][n % 7])
        ax.plot(xx, y_n, ls="--", lw=1.2, color=col)
        # l'etichetta va sopra i DATI attorno al picco, non sopra la sola gaussiana:
        # se no finisce dentro l'istogramma e copre proprio il picco che indica
        vicino = np.abs(centers - m_tutti[n]) < max(q_pe / 4, w)
        y_lab = max(y_n.max(), counts[vicino].max() if vicino.any() else 0)
        etichette.append((f"{n}", m_tutti[n], y_lab, col))
    ax.plot([], [], ls="--", color="gray", label="single peaks (fit parameters); number = p.e.")

    ax.set_xlabel("Charge [ADC]")
    ax.set_ylabel(f"Events / {w:.1f} ADC")
    ax.set_title(f"Fit ({MODELLO_BUIO}) — {DESCRIZIONE[lab]}" + ("" if affidabile else "  (NOT RELIABLE)"))
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.20)     # spazio per le etichette del picco piu' alto
    ax.legend(fontsize=8)
    plt.tight_layout()
    posa_etichette(ax, etichette, centers, counts)
    plt.show()

# --- Riepilogo su tutti i run ---
if fit_gain2:
    print(f"\n{'run':>8} {'Q_pe [ADC]':>16} {'lambda':>17} {'1/(1-lam)':>10} {(PAR3[0] if PAR3 else '-'):>14} {'f0':>7} {'chi2/ndf':>9}")
    for l in fit_gain2:
        r = fit_gain2[l]
        print(f"{DESCRIZIONE[l]:>8} {r['q_pe']:9.1f} ± {r['e_q_pe']:5.1f} {r['lam']:8.4f} ± {r['e_lam']:6.4f} "
              f"{r['celle_medie']:10.3f} {r['par3']:6.3f} ± {r['e_par3']:5.3f} {r['f0']:7.3f} {r['chi2_rid']:9.2f}"
              + ("" if r["affidabile"] else "  Q_pe non affidabile"))


In [ ]:
# === Confronto: gate stretto attorno al picco vs gate su tutto il segnale ===
# Gate "intero": dalla salita alla fine della coda, presi dalla waveform media
# di ciascun run.
#  - inizio: dove la media supera il 10% dell'ampiezza, meno un margine, per
#    non perdere gli eventi che salgono prima (l'inizio si sposta da evento a evento);
#  - fine: ultimo campione in cui la media e' ancora sopra il 10% dell'ampiezza.
# La baseline per evento deve finire prima del gate, se no sottrae segnale.
MARGINE_INIZIO_NS = 20
FRAZIONE_CODA = 0.10


def binnaggio_fd(x, lo, hi):
    """Bordi dei bin con Freedman-Diaconis sugli eventi in [lo, hi]."""
    x = x[(x >= lo) & (x <= hi)]
    q25, q75 = np.percentile(x, [25, 75])
    n = max(1, int(np.ceil((hi - lo) / (2 * (q75 - q25) / len(x) ** (1 / 3)))))
    return np.linspace(lo, hi, n + 1)


charge_intero_gain2 = {}
for lab in wf_analisi:
    media = wf_media_gain2[lab]
    bl_m = media[(t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])].mean()
    amp_m = media.max() - bl_m
    # Ai bordi della traccia il filtro (filtfilt) lascia oscillazioni che superano
    # il 10%. Per questo l'inizio si cerca partendo dal massimo e andando indietro
    # fino al primo campione sotto il 10%: cosi' non dipende da dove comincia la
    # baseline (con BASELINE_NS da 0 ns la ricerca in avanti finiva sul bordo).
    # La fine e' l'ultimo campione sopra il 10%, esclusi gli ultimi 20 ns.
    sopra = media - bl_m > FRAZIONE_CODA * amp_m
    i_max = int(np.argmax(media))
    sotto_prima = np.flatnonzero(~sopra[:i_max])
    i_salita = sotto_prima[-1] + 1 if len(sotto_prima) else 0
    zona = t <= t[-1] - 20
    i_fine = np.flatnonzero(sopra & zona)[-1]
    gate_intero = (t[i_salita] - MARGINE_INIZIO_NS, t[i_fine])
    baseline_intero = (BASELINE_NS[0], min(BASELINE_NS[1], gate_intero[0] - 10))
    if np.count_nonzero((t >= baseline_intero[0]) & (t <= baseline_intero[1])) < 10:
        raise ValueError(f"{lab} V: la baseline ({baseline_intero[0]:.0f}, {baseline_intero[1]:.0f}) ns "
                         f"prima del gate intero e' vuota o troppo corta: anticipare BASELINE_NS")

    charge_intero = compute_charge(wf_analisi[lab], gate_intero, baseline_intero)
    charge_intero_gain2[lab] = charge_intero
    charge_stretto = charge_gain2[lab]

    print(f"=== {lab} V ===")
    print(f"Gate stretto: {GATE_NS} ns, baseline {BASELINE_NS} ns")
    print(f"Gate intero : ({gate_intero[0]:.0f}, {gate_intero[1]:.0f}) ns, "
          f"baseline ({baseline_intero[0]:.0f}, {baseline_intero[1]:.0f}) ns")

    if not mostra(lab):
        continue

    # Intervallo di ciascun istogramma dai percentili: le due cariche hanno scale diverse
    fig = plt.figure(figsize=(16, 9))
    ax0 = fig.add_subplot(2, 1, 1)
    ax0.plot(t, media, color="k", label="mean waveform")
    ax0.axvspan(*GATE_NS, color="orange", alpha=0.35, label=f"narrow gate {GATE_NS} ns")
    ax0.axvspan(*gate_intero, color="blue", alpha=0.10,
                label=f"full gate ({gate_intero[0]:.0f}, {gate_intero[1]:.0f}) ns")
    ax0.axvspan(*baseline_intero, color="green", alpha=0.15, label="baseline")
    ax0.set_xlabel("Time [ns]"); ax0.set_ylabel("ADC"); ax0.grid(True, alpha=0.3); ax0.legend(fontsize=8)

    for k, (q, nome, colore) in enumerate([(charge_stretto, "narrow gate", "darkorange"),
                                           (charge_intero, "full gate", "blue")]):
        ax = fig.add_subplot(2, 2, 3 + k)
        lo, hi = np.percentile(q, [0.1, 99.9])
        edges = binnaggio_fd(q, lo, hi)
        ax.hist(q, bins=edges, histtype="step", color=colore)
        ax.set_title(f"{nome} — {len(edges) - 1} bins of {edges[1] - edges[0]:.1f} ADC")
        ax.set_xlabel("Charge [ADC]"); ax.set_ylabel("Events"); ax.grid(True, alpha=0.3)
    fig.suptitle(f"{lab} V — charge ({'filtered' if USA_FILTRATE else 'raw'} waveforms)")
    plt.tight_layout()
    plt.show()

In [ ]:
# === Waveform che formano ciascun picco della distribuzione di carica ===
# Come nel notebook Gain, con i pesi del fit di Borel (fit_gain2).
# Le finestre dei picchi vengono dal fit,
# non dall'istogramma: ogni picco n p.e. e' la gaussiana disegnata nella cella
# del fit (peso, posizione e larghezza dal fit). La finestra del picco n e'
# dove la sua gaussiana e' la piu' alta di tutte: li' un evento ha piu'
# probabilita' di essere n p.e. che qualsiasi altra cosa, ed e' la divisione
# che sbaglia meno. Un picco che non e' mai il piu' alto (a 52 V succede ai
# picchi coperti dal piedistallo) non ha finestra e non si mostra.
#
# Dall'area delle gaussiane dentro ogni finestra:
#  - purezza:    frazione degli eventi nella finestra che sono davvero n p.e.
#  - efficienza: frazione degli eventi n p.e. che cadono nella finestra
# Con picchi sovrapposti la purezza scende, e la waveform media della finestra
# mescola picchi diversi: va letta insieme alla purezza.
# Si usa la carica del gate stretto (charge_gain2), quella del fit.
# Solo visualizzazione: non cambia nulla delle celle precedenti.
from scipy.special import erf

# Si mostrano tutti i picchi con almeno MIN_EVENTI_PICCO eventi attesi: e' la
# statistica, non un numero fisso, a decidere fin dove arrivare. N_PICCHI_MAX serve solo a tagliare i grafici
# quando sono troppi; None = nessun taglio.
N_PICCHI_MAX = None           # picchi mostrati per run, al massimo; None = tutti
MIN_EVENTI_PICCO = 100        # picchi con meno eventi attesi nella finestra non si mostrano
MAX_WF_PER_PICCO = 1000       # linee disegnate per picco (a caso); None = tutte (lento)
SEED_PICCHI = 0


def finestre_dal_fit(r, x_min, x_max):
    """[(n, inizio, fine)] delle regioni in cui la gaussiana del picco n e' la piu' alta."""
    pesi, s0, s1 = r["pesi"], r["sigma0"], r["sigma1"]
    m_tutti = posizioni_picchi(r["posizioni"], r["n_tot"])
    xx = np.linspace(x_min, x_max, 20000)
    comp = np.array([picco_singolo(xx, n, pesi, s0, s1, m_tutti[n]) for n in range(r["n_tot"])])
    vince = np.argmax(comp, axis=0)
    cambi = np.flatnonzero(np.diff(vince)) + 1
    inizi = np.r_[0, cambi]
    fini = np.r_[cambi, len(xx)]
    finestre = []
    for i0, i1 in zip(inizi, fini):
        a = -np.inf if i0 == 0 else 0.5 * (xx[i0 - 1] + xx[i0])
        b = np.inf if i1 == len(xx) else 0.5 * (xx[i1 - 1] + xx[i1])
        finestre.append((int(vince[i0]), a, b))
    return finestre


def frazione_gaussiana(a, b, m, s):
    """Frazione di una gaussiana (m, s) dentro [a, b)."""
    cdf = lambda x: 0.5 * (1 + erf((x - m) / (s * np.sqrt(2))))
    return cdf(b) - cdf(a)


finestre_gain2 = {}
for lab_p in fit_gain2:
    r = fit_gain2[lab_p]
    q_p = charge_gain2[lab_p]
    wf_p = wf_analisi[lab_p]
    h = hist_gain2[lab_p]
    pesi, s0, s1 = r["pesi"], r["sigma0"], r["sigma1"]
    m_tutti = posizioni_picchi(r["posizioni"], r["n_tot"])
    sig = lambda n: np.sqrt(s0 ** 2 + n * s1 ** 2)
    n_eventi = len(q_p)

    finestre = []
    for n, a, b in finestre_dal_fit(r, q_p.min(), q_p.max()):
        # eventi attesi nella finestra, picco per picco: N * P(k) * frazione della gaussiana k dentro
        attesi = np.array([n_eventi * peso_buio(k, *pesi) * frazione_gaussiana(a, b, m_tutti[k], sig(k))
                           for k in range(r["n_tot"])])
        osservati = np.count_nonzero((q_p >= a) & (q_p < b))
        # finestre lontane da tutto (coda estrema) hanno area attesa nulla: niente purezza
        if attesi.sum() <= 0:
            continue
        finestre.append({"n": n, "a": a, "b": b, "osservati": osservati, "attesi_n": attesi[n],
                         "purezza": attesi[n] / attesi.sum(),
                         "efficienza": frazione_gaussiana(a, b, m_tutti[n], sig(n))})
    finestre = [f for f in finestre if f["attesi_n"] >= MIN_EVENTI_PICCO][:N_PICCHI_MAX]
    if not finestre:
        print(f"=== {DESCRIZIONE[lab_p]} === nessun picco con almeno {MIN_EVENTI_PICCO} eventi attesi")
        continue
    finestre_gain2[lab_p] = finestre

    print(f"=== {DESCRIZIONE[lab_p]} ===" + ("" if r["affidabile"] else "  (fit non affidabile)"))
    print(f"{'picco':>7} {'finestra [ADC]':>22} {'eventi':>8} {'attesi n p.e.':>14} {'purezza':>8} {'efficienza':>11}")
    for f in finestre:
        print(f"{f['n']:>5} pe [{f['a']:9.0f}, {f['b']:9.0f}) {f['osservati']:8d} {f['attesi_n']:14.0f} "
              f"{100 * f['purezza']:7.1f}% {100 * f['efficienza']:10.1f}%")

    if not mostra(lab_p):
        continue
    col_pe = lambda n: plt.cm.tab10([1, 2, 4, 5, 6, 8, 9][n % 7])   # come nella cella del fit

    # --- Panoramica: istogramma, fit, gaussiane e finestre ---
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.step(h["centers"], h["counts"], where="mid", color="tab:blue", label=DESCRIZIONE[lab_p])
    xx = np.linspace(h["centers"][0], h["centers"][-1], 2000)
    norm = h["counts"].sum() * h["bin_width"]
    ax.plot(xx, sum(picco_singolo(xx, k, pesi, s0, s1, m_tutti[k]) for k in range(r["n_tot"])) * norm,
            color="red", label="Borel fit")
    for f in finestre:
        n = f["n"]
        ax.plot(xx, picco_singolo(xx, n, pesi, s0, s1, m_tutti[n]) * norm, ls="--", color=col_pe(n))
        a = max(f["a"], h["centers"][0]); b = min(f["b"], h["centers"][-1])
        ax.axvspan(a, b, color=col_pe(n), alpha=0.15,
                   label=f"{n} p.e.: {f['osservati']} events, purity {100 * f['purezza']:.0f}%, "
                         f"efficiency {100 * f['efficienza']:.0f}%")
    ax.set_xlabel("Charge [ADC]"); ax.set_ylabel(f"Events / {h['bin_width']:.1f} ADC")
    ax.set_title(f"{DESCRIZIONE[lab_p]} — peak windows from the fit (gate {GATE_NS} ns)")
    ax.grid(True, alpha=0.3)
    # legenda fuori: con 10-15 picchi dentro coprirebbe la distribuzione
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    plt.tight_layout()
    plt.show()

    # --- Un grafico per picco: waveform e loro media, stessa scala y per tutti i picchi del run ---
    rng_p = np.random.default_rng(SEED_PICCHI)
    y_lo, y_hi = np.percentile(wf_p[:2000], [0.1, 99.9])
    medie = {}
    for f in finestre:
        n = f["n"]
        sel = np.flatnonzero((q_p >= f["a"]) & (q_p < f["b"]))
        if len(sel) == 0:
            continue
        medie[n] = wf_p[sel].mean(axis=0)
        disegna = sel if MAX_WF_PER_PICCO is None or len(sel) <= MAX_WF_PER_PICCO \
            else rng_p.choice(sel, MAX_WF_PER_PICCO, replace=False)
        fig, ax = plt.subplots(figsize=(15, 4.5))
        ax.plot(t, wf_p[disegna].T, color=col_pe(n), lw=0.3, alpha=0.08, rasterized=True)
        ax.plot(t, medie[n], color="k", lw=2, label=f"mean over all {len(sel)} events")
        ax.axvspan(*BASELINE_NS, color="green", alpha=0.12, label=f"baseline {BASELINE_NS} ns")
        ax.axvspan(*GATE_NS, color="orange", alpha=0.25, label=f"gate {GATE_NS} ns")
        ax.set_ylim(y_lo, y_hi)
        ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
        ax.set_title(f"{DESCRIZIONE[lab_p]} — {n} p.e. (purity {100 * f['purezza']:.0f}%): "
                     f"{len(disegna)} of {len(sel)} waveforms drawn")
        plt.tight_layout()
        plt.show()

    # --- Le medie dei picchi sovrapposte: l'ampiezza deve crescere di un passo per p.e. ---
    if medie:
        fig, ax = plt.subplots(figsize=(15, 4.5))
        # scala di colore ordinata: i 7 colori di col_pe si ripeterebbero oltre 6 p.e.
        n_max = max(medie)
        for n, med in medie.items():
            ax.plot(t, med, color=plt.cm.viridis(n / max(n_max, 1)), lw=1.5, label=f"{n} p.e.")
        ax.axvspan(*GATE_NS, color="orange", alpha=0.15, label=f"gate {GATE_NS} ns")
        ax.set_xlabel("Time [ns]"); ax.set_ylabel("ADC"); ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
        ax.set_title(f"{DESCRIZIONE[lab_p]} — mean waveform of each peak")
        plt.tight_layout()
        plt.show()


In [ ]:
# === Impulsi al buio che il trigger non ha scelto: trigger software prima del trigger ===
# Il fit di Borel sull'impulso del trigger trova troppo pochi eventi a 1 p.e.
# (eps1 ~0.37). Per sapere se il difetto e' nel SiPM o nella selezione del
# trigger si guardano gli impulsi che cadono PRIMA del trigger (che sale a
# ~452 ns): li' il trigger hardware non ha scelto niente.
#
# Prima prova, scartata: il gate fisso del LED, (220, 270) ns, su quella zona.
# Gli impulsi a caso non sono allineati al gate: uno che comincia a meta' gate
# porta dentro meta' carica, uno caduto prima porta la sua coda. Il continuo
# riempie le valli: Q_pe usciva del 10% sotto al LED e lambda non si misurava
# (0 ± 0.1). Col LED il problema non c'e', il lampo e' sempre nello stesso punto.
#
# Qui invece ogni impulso si trova con una soglia software (primo campione
# sopra SOGLIA_RMS volte il rumore) e gate e baseline si mettono rispetto a
# quel campione come nelle altre due analisi: gate da 12 a 49 campioni dopo
# (38 campioni), baseline da 153 a 18 campioni prima (136 campioni).
# Nota: col LED e sul trigger il riferimento e' la salita a 5 sigma; con una
# soglia piu' bassa il campione trovato cade un po' prima, cioe' il gate si
# sposta al massimo di un paio di campioni verso l'inizio dell'impulso.
#
# Il punto della misura e' che la soglia la si sceglie e la si fa variare:
# se il picco a 1 p.e. si svuota alzando la soglia software come lo vuota il
# trigger hardware, il difetto e' la soglia, e la si misura in p.e.
# Il fit e' quello di Borel con eps1 (cella del fit): con una soglia
# trascurabile eps1 deve tornare ~1 e lambda deve essere quello del LED.
# f0 raccoglie le soglie superate dal rumore, che nel gate non hanno impulsi.
# STATO AL 2026-10-14: ESPLORATIVA, i numeri non sono ancora una misura.
#  - eps1 dipende molto dalla soglia software (a 54 V: 0.39 a 4 rms, 0.15 a 5,
#    0.07 a 6), come ci si aspetta se il picco a 1 p.e. lo taglia una soglia;
#    lambda pero' resta sotto il LED (0.09-0.14 contro 0.141 a 54 V) e Q_pe
#    salta fra una soglia e l'altra (a 53.5 V 182 e 292 ADC).
#  - Le soglie basse scattano sul rumore, e quegli eventi formano un picco a
#    ~+100 ADC, non a 0: il modello mette il piedistallo un passo sotto il
#    picco a 1 p.e. e confonde rumore e 1 p.e. Va liberata la sua posizione.
#  - L'rms cresce con la tensione (2.1 ADC a 53.5 V, 4.4 a 56.5): lo gonfiano
#    gli impulsi al buio stessi. Le soglie in rms non sono le stesse in p.e.
#  - Non verificato: se la scheda era pronta a triggerare gia' prima del
#    trigger, gli impulsi sopra la soglia hardware li' mancano, e lambda di
#    questa cella esce basso per questo.
SOGLIE_RMS = [3, 4, 5, 6]          # soglie software, in rms del rumore
SOGLIA_GRAFICI = 3                  # soglia di cui si disegnano i fit
CERCA_DA_NS, CERCA_A_NS = 204, 364  # dove si cercano gli impulsi: baseline intera dopo 0 ns,
                                    # gate finito prima della salita del trigger (~430 ns al 16%)
GATE_DOPO, GATE_LUNGO = 12, 38      # [campioni] dal campione della soglia
BL_PRIMA, BL_LUNGA = 153, 136       # [campioni] la baseline comincia BL_PRIMA prima

c0, c1 = int(np.ceil(CERCA_DA_NS / SAMPLING_TIME)), int(CERCA_A_NS / SAMPLING_TIME)
if c0 < BL_PRIMA or c1 + GATE_DOPO + GATE_LUNGO > N_SAMPLES:
    raise ValueError("finestra di ricerca incompatibile con baseline o gate: rivedere CERCA_DA/A_NS")


def impulsi_software(wf, soglia_rms):
    """Primo impulso sopra soglia nella zona di ricerca, evento per evento.

    Restituisce (carica nel gate allineato, maschera degli eventi con un impulso
    trovato e la baseline pulita, rms del rumore). La soglia si applica sulla
    traccia meno la mediana dei primi 200 ns; la carica usa la baseline del
    singolo impulso, come compute_charge.
    """
    x = wf.astype(np.float64)
    rif = np.median(x[:, : int(200 / SAMPLING_TIME)], axis=1, keepdims=True)
    x = x - rif
    rms = 1.4826 * np.median(np.median(np.abs(x[:, : int(200 / SAMPLING_TIME)]), axis=1))
    sopra = x[:, c0:c1 + 1] > soglia_rms * rms
    # fronte: sopra adesso, sotto al campione prima (un impulso gia' alto a c0 non conta)
    prima = x[:, c0 - 1:c1] > soglia_rms * rms
    fronte = sopra & ~prima
    trovato = fronte.any(axis=1)
    c = c0 + np.argmax(fronte, axis=1)
    cs = np.concatenate([np.zeros((len(x), 1)), np.cumsum(x, axis=1)], axis=1)
    righe = np.arange(len(x))
    somma = lambda a, n: cs[righe, a + n] - cs[righe, a]      # somma di n campioni da a
    bl = somma(c - BL_PRIMA, BL_LUNGA) / BL_LUNGA
    q = somma(c + GATE_DOPO, GATE_LUNGO) - bl * GATE_LUNGO
    # baseline pulita: nessun campione sopra 4 rms (come il taglio sulla baseline del trigger)
    idx_bl = (c - BL_PRIMA)[:, None] + np.arange(BL_LUNGA)
    max_bl = (x[righe[:, None], idx_bl] - bl[:, None]).max(axis=1)
    buoni = trovato & (max_bl < N_RMS_BASELINE * rms)
    return q, buoni, rms


def fit_borel_eps1(counts, centers, w, passo0):
    """Fit di Borel con eps1 di un istogramma (stesso modello della cella del fit)."""
    m0 = semina_picchi(counts, centers, passo0)
    K = len(m0)
    if K < 2:
        return None
    n_tot = K + 1
    while n_tot < 60 and 1 - sum(borel_pmf(n, 0.3) for n in range(1, n_tot)) > P_CODA:
        n_tot += 1
    norm = counts.sum() * w
    dens, err = counts / norm, np.sqrt(np.maximum(counts, 1)) / norm
    f = lambda x, *p: densita_picchi_liberi(x, p[:3], p[3], p[4], np.array(p[5:]), n_tot)
    p0 = [0.1, 0.15, 0.9, passo0 / 4, passo0 / 8, *m0]
    lim = ([0, 0, 0.01, 1, 0, *(m0 - passo0 / 2)], [0.95, 0.9, 1.0, 2 * passo0, 2 * passo0, *(m0 + passo0 / 2)])
    popt, pcov = curve_fit(f, centers, dens, p0=p0, sigma=err, bounds=lim, maxfev=50000)
    chi2_rid = np.sum(((dens - f(centers, *popt)) / err) ** 2) / (len(centers) - len(popt))
    e = np.sqrt(np.diag(pcov)) * np.sqrt(max(chi2_rid, 1.0))
    m = popt[5:]
    return {"f0": popt[0], "lam": popt[1], "e_lam": e[1], "eps1": popt[2], "e_eps1": e[2],
            "sigma0": popt[3], "sigma1": popt[4], "q_pe": np.diff(m).mean(), "posizioni": m,
            "n_tot": n_tot, "chi2_rid": chi2_rid, "popt": popt, "f": f, "norm": norm}


if MODELLO_BUIO != "eps1":
    raise ValueError('questa cella usa il modello "eps1": impostare MODELLO_BUIO = "eps1" nella cella della distribuzione')

fit_software = {}         # (label, soglia) -> risultati
for lab in wf_tutte:
    if lab not in fit_gain2:
        print(f"[{DESCRIZIONE[lab]}] niente fit di Borel sul trigger: manca il passo iniziale, si salta")
        continue
    for k in SOGLIE_RMS:
        q, buoni, rms = impulsi_software(wf_tutte[lab], k)
        qb = q[buoni]
        lo, hi = qb.min(), qb.max()
        q25, q75 = np.percentile(qb, [25, 75])
        n_bins = max(1, int(np.ceil((hi - lo) / (2 * (q75 - q25) / len(qb) ** (1 / 3)))))
        edges = np.linspace(lo, hi, n_bins + 1)
        counts, _ = np.histogram(qb, bins=edges)
        centers, w = 0.5 * (edges[:-1] + edges[1:]), edges[1] - edges[0]
        try:
            r = fit_borel_eps1(counts, centers, w, fit_gain2[lab]["q_pe"])
        except RuntimeError as e:
            r = None
        if r is None:
            print(f"[{DESCRIZIONE[lab]}] soglia {k} rms: fit non riuscito")
            continue
        r.update({"n_impulsi": len(qb), "soglia_adc": k * rms})
        fit_software[(lab, k)] = r

        if not (mostra(lab) and k == SOGLIA_GRAFICI):
            continue
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.step(centers, counts, where="mid", color="tab:blue",
                label=f"{DESCRIZIONE[lab]}, software threshold {k} rms = {k * rms:.1f} ADC ({len(qb)} pulses)")
        xx = np.linspace(centers[0], centers[-1], 2000)
        ax.plot(xx, r["f"](xx, *r["popt"]) * r["norm"], color="red",
                label=f"Borel + $\\epsilon_1$: $\\lambda$ = {r['lam']:.3f}, $\\epsilon_1$ = {r['eps1']:.2f}")
        m_tutti = posizioni_picchi(r["posizioni"], r["n_tot"])
        for n in range(r["n_tot"]):
            y_n = picco_singolo(xx, n, tuple(r["popt"][:3]), r["sigma0"], r["sigma1"], m_tutti[n]) * r["norm"]
            if y_n.max() >= 1:
                ax.plot(xx, y_n, ls="--", lw=1.2, color=plt.cm.tab10([1, 2, 4, 5, 6, 8, 9][n % 7]))
        ax.set_yscale("log"); ax.set_ylim(0.5, counts.max() * 3)
        ax.set_xlabel("Charge [ADC]"); ax.set_ylabel(f"Events / {w:.1f} ADC")
        ax.set_title(f"Dark pulses before the trigger, gate aligned on each pulse — {DESCRIZIONE[lab]}")
        ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

# --- Confronto con il trigger hardware e con il LED ---
# Valori del LED dal notebook Gain (fit del 2026-10-07), ricopiati a mano:
# da aggiornare se quel fit cambia.
LAMBDA_LED = {"52.5": 0.019, "53": 0.057, "53.5": 0.100, "54": 0.141, "54.5": 0.192,
              "55": 0.240, "55.5": 0.290, "56": 0.357, "56.5": 0.411}
QPE_LED = {"52.5": 152.1, "53": 222.0, "53.5": 291.4, "54": 351.6, "54.5": 431.4,
           "55": 508.8, "55.5": 580.9, "56": 650.5, "56.5": 736.3}
print(f"{'run':>8} {'soglia':>14} {'impulsi':>8} {'lambda':>17} {'eps1':>15} {'Q_pe':>7} {'chi2/ndf':>9}")
for lab in fit_gain2:
    for k in SOGLIE_RMS:
        if (lab, k) not in fit_software:
            continue
        r = fit_software[(lab, k)]
        print(f"{DESCRIZIONE[lab]:>8} {k:3d} rms {r['soglia_adc']:5.1f} ADC {r['n_impulsi']:8d} "
              f"{r['lam']:8.4f} ± {r['e_lam']:.4f} {r['eps1']:6.3f} ± {r['e_eps1']:.3f} {r['q_pe']:7.1f} {r['chi2_rid']:9.2f}")
    hw = fit_gain2[lab]
    print(f"{'':>8} {'trigger HW':>14} {'':>8} {hw['lam']:8.4f} ± {hw['e_lam']:.4f} {hw['par3']:6.3f} ± {hw['e_par3']:.3f} "
          f"{hw['q_pe']:7.1f} {hw['chi2_rid']:9.2f}")
    print(f"{'':>8} {'LED':>14} {'':>8} {LAMBDA_LED.get(lab, np.nan):8.4f} {'':>9} {'':>15} {QPE_LED.get(lab, np.nan):7.1f}\n")


In [ ]:
# === Breakdown dalla retta Q_pe(V) al buio, confrontato con quello del LED ===
# Stesso metodo del notebook Gain: retta Q_pe = k * (V - V_bd) pesata con gli
# errori, errori allargati di sqrt(chi2) se gli scarti superano gli errori,
# V_bd = -b/k con la covarianza. Q_pe qui e' quello del fit di Borel (eps1):
# la distanza fra i picchi non dipende da come il trigger ha scelto gli eventi.
# Serve a sapere se le run al buio (2026-10-06) e col LED (2026-10-07) sono
# confrontabili: V_bd si sposta di ~54 mV/°C, e una differenza di temperatura
# fra i due giorni si vedrebbe qui.
# Il confronto si fa due volte: con la retta del Gain (tutte le sue tensioni,
# 52-56.5 V) e con una retta del LED rifatta sulle sole tensioni del buio, per
# non scambiare una differenza di intervallo per una differenza fra i due giorni.
# Q_pe del LED dal notebook Gain (fit del 2026-10-07), ricopiati a mano.
E_QPE_LED = {"52.5": 1.9, "53": 1.4, "53.5": 2.2, "54": 4.0, "54.5": 4.5,
             "55": 4.5, "55.5": 7.0, "56": 9.2, "56.5": 15.1}
K_LED_GAIN, VBD_LED_GAIN, E_VBD_LED_GAIN = 141.5, 51.433, 0.021   # retta del Gain, 52-56.5 V


def retta_breakdown(v, q, eq):
    """(k, b, V_bd, errore su V_bd, chi2/ndf) della retta q = k*v + b pesata con 1/eq."""
    (k, b), cov = np.polyfit(v, q, 1, w=1 / eq, cov="unscaled")
    chi2 = np.sum(((q - (k * v + b)) / eq) ** 2) / (len(v) - 2)
    cov = cov * max(chi2, 1.0)
    grad = np.array([b / k ** 2, -1 / k])
    return k, b, -b / k, np.sqrt(grad @ cov @ grad), chi2


labs_bd = [l for l in fit_gain2 if fit_gain2[l]["affidabile"]]
if len(labs_bd) < 3:
    raise ValueError(f"solo {len(labs_bd)} run affidabili: troppo pochi per una retta")
v_b = np.array([VBIAS[l] for l in labs_bd])
q_b = np.array([fit_gain2[l]["q_pe"] for l in labs_bd])
e_b = np.array([fit_gain2[l]["e_q_pe"] for l in labs_bd])
k_b, b_b, vbd_b, e_vbd_b, chi2_b = retta_breakdown(v_b, q_b, e_b)

labs_led = [l for l in labs_bd if l in QPE_LED]
v_l = np.array([VBIAS[l] for l in labs_led])
q_l = np.array([QPE_LED[l] for l in labs_led])
e_l = np.array([E_QPE_LED[l] for l in labs_led])
k_l, b_l, vbd_l, e_vbd_l, chi2_l = retta_breakdown(v_l, q_l, e_l)

print(f"Run usati: {', '.join(DESCRIZIONE[l] for l in labs_bd)}")
print(f"  buio          : {k_b:6.1f} ADC/V,  V_bd = {vbd_b:.3f} ± {e_vbd_b:.3f} V,  chi2/ndf = {chi2_b:.2f}")
print(f"  LED, stesse V : {k_l:6.1f} ADC/V,  V_bd = {vbd_l:.3f} ± {e_vbd_l:.3f} V,  chi2/ndf = {chi2_l:.2f}")
print(f"  LED, Gain     : {K_LED_GAIN:6.1f} ADC/V,  V_bd = {VBD_LED_GAIN:.3f} ± {E_VBD_LED_GAIN:.3f} V  (52-56.5 V)")
d_vbd = vbd_b - vbd_l
e_d = np.hypot(e_vbd_b, e_vbd_l)
print(f"Differenza V_bd buio - LED (stesse V): {1000 * d_vbd:+.0f} ± {1000 * e_d:.0f} mV "
      f"({abs(d_vbd) / e_d:.1f} sigma); a 54 mV/°C equivale a {d_vbd / 0.054:+.1f} ± {e_d / 0.054:.1f} °C")
print(f"Pendenze: buio/LED = {k_b / k_l:.3f}  (stesse V)")

fig, (ax, axr) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(v_b, q_b, e_b, fmt="o", color="k", label="dark (Borel fit), 2026-10-06")
ax.errorbar(v_l + 0.03, q_l, e_l, fmt="s", color="tab:blue", mfc="white", label="LED (generalized Poisson), 2026-10-07")
xr = np.linspace(min(vbd_b, vbd_l) - 0.1, v_b.max() + 0.2, 100)
ax.plot(xr, k_b * xr + b_b, color="k", label=f"dark: $V_{{bd}}$ = {vbd_b:.2f} ± {e_vbd_b:.2f} V")
ax.plot(xr, k_l * xr + b_l, color="tab:blue", ls="--",
        label=f"LED, same voltages: $V_{{bd}}$ = {vbd_l:.2f} ± {e_vbd_l:.2f} V")
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("$Q_{pe}$ in the gate [ADC]")
ax.set_title("Breakdown voltage: dark vs LED")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
axr.errorbar(v_b, q_b - (k_l * v_b + b_l), e_b, fmt="o", color="k", label="dark - LED line")
axr.errorbar(v_l + 0.03, q_l - (k_l * v_l + b_l), e_l, fmt="s", color="tab:blue", mfc="white", label="LED - LED line")
axr.axhline(0, color="gray", lw=0.8)
axr.set_xlabel("Bias voltage [V]"); axr.set_ylabel("Residual [ADC]")
axr.grid(True, alpha=0.3); axr.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# === Q_pe dell'impulso intero da una cella sola, confrontato col LED ===
# Nel notebook Gain Q_pe_tot = R * Q_pe, con R = carica su tutto l'impulso /
# carica nel gate, misurato sulla waveform MEDIA del LED. Quella forma contiene
# anche come il LED emette nel tempo: nelle run col LED si vedono lampi
# ritardati a gruppi (~260, 320, 360 ns), luce che cade fuori dal gate e
# dentro la finestra lunga. Se pesa, R e Q_pe_tot sono sovrastimati e il light
# yield dei pannelli verrebbe sottostimato. Al buio ogni impulso e' una cella:
# e' la risposta del SiPM a un fotoelettrone, senza il LED di mezzo.
#
# Metodo: differenza fra picchi. La carica su tutto l'impulso di un evento al
# buio contiene anche gli impulsi a caso che seguono quello del trigger (~1.5
# MHz per ~900 ns: piu' di uno in media, quanto il segnale), e non si toglie
# con la baseline, che e' scelta pulita. Ma e' la stessa negli eventi da 1, 2 e
# 3 p.e.: la differenza fra le cariche medie di due picchi vicini e' la carica
# di UNA cella in piu' (con i suoi afterpulse, che fanno parte della risposta
# anche nei pannelli). Sulla finestra lunga e' Q_pe_tot, misurato direttamente;
# divisa per la stessa differenza nel gate da' R al buio.
# Gli eventi di ogni picco sono quelli delle finestre del fit (cella delle
# finestre): la purezza non e' 100%, ma i picchi mescolati spostano le due
# cariche nello stesso modo e il rapporto ne risente poco.
# La differenza di tempo degli 1 p.e. (salgono ~15 ns prima, scelta del
# trigger) sposta un pezzo di coda oltre la fine della traccia: per questo si
# confronta anche 3 - 2, dove i due picchi hanno la stessa forma.
#
# Finestra lunga: dalla fine della baseline alla fine della traccia, come nel
# Gain. Al buio pero' la traccia finisce ~896 ns dopo la salita (PostTrigger
# 40), col LED ~1144 ns (PostTrigger 80): il LED si ricalcola anche con la
# finestra accorciata alla stessa durata, ed e' quello il confronto alla pari.
# RISULTATO (2026-10-14): il timore sul LED non e' confermato, ma R al buio
# con questo metodo NON e' una misura.
#  - R del LED non cambia accorciando la finestra a 896 ns (4.22 contro 4.20 a
#    54 V): oltre quel punto la luce ritardata del LED non porta carica, e la
#    coda della media del LED non sta sopra quella della cella singola.
#  - R al buio esce 5.8 (2-1) e 5.1 (3-2) a 54 V, 10.7 e 5.9 a 56.5 V, contro
#    4.2 del LED, e il 2-1 cresce con la tensione come il DCR. La cancellazione
#    del fondo non regge: scegliere gli eventi per la carica NEL GATE sceglie
#    anche quelli con un impulso in piu' vicino al gate (afterpulse, crosstalk
#    ritardato, impulsi a caso), che entra in parte nel gate e tutto nella
#    finestra lunga. Il LED non ha questo difetto: li' R e' il rapporto delle
#    cariche medie su TUTTI gli eventi, senza scelta.
#  - Per il light yield resta Q_pe_tot del LED. La risposta della cella singola
#    si potra' misurare sugli impulsi al buio isolati nelle run dei pannelli,
#    col trigger esterno (nessuna scelta sugli impulsi).
#  - La media del LED ha gobbe a ~60 ns l'una dall'altra, la cella singola no:
#    il LED non emette in un colpo solo (lo dicevano anche i lampi ritardati).
DIR_LED = os.path.abspath(os.path.join("..", "data/10072026/SiPM_001"))
GATE_LED_NS, BASELINE_LED_NS = (220, 270), (0, 180)   # quelli del notebook Gain
SALITA_LED_NS, SALITA_BUIO_NS = 204.0, 452.0          # inizio salita mediano (cella del gate)
MAX_EVENTI_LED = 50000

fine_buio = t[-1]
durata_dopo_salita = fine_buio - SALITA_BUIO_NS                 # ~896 ns
FINESTRA_BUIO_NS = (BASELINE_NS[1], fine_buio)
FINESTRA_LED_PIENA_NS = (BASELINE_LED_NS[1], t[-1])
FINESTRA_LED_CORTA_NS = (BASELINE_LED_NS[1], SALITA_LED_NS + durata_dopo_salita)

ris_qtot = {}
for lab in fit_gain2:
    if not fit_gain2[lab]["affidabile"] or lab not in finestre_gain2:
        continue
    q_gate = charge_gain2[lab]
    q_full = compute_charge(wf_analisi[lab], FINESTRA_BUIO_NS, BASELINE_NS)
    medie = {}
    for f in finestre_gain2[lab]:
        if f["n"] in (1, 2, 3):
            sel = (q_gate >= f["a"]) & (q_gate < f["b"])
            if sel.sum() >= 50:
                cv = np.cov(q_full[sel], q_gate[sel]) / sel.sum()
                medie[f["n"]] = (q_full[sel].mean(), q_gate[sel].mean(), cv, sel.sum())
    r = {"medie": medie}
    for a, b in ((1, 2), (2, 3)):
        if a in medie and b in medie:
            d_full = medie[b][0] - medie[a][0]
            d_gate = medie[b][1] - medie[a][1]
            # errori: le due finestre sono eventi diversi, quindi indipendenti
            v_full = medie[b][2][0, 0] + medie[a][2][0, 0]
            v_gate = medie[b][2][1, 1] + medie[a][2][1, 1]
            c_fg = medie[b][2][0, 1] + medie[a][2][0, 1]
            R = d_full / d_gate
            e_R = abs(R) * np.sqrt(v_full / d_full ** 2 + v_gate / d_gate ** 2 - 2 * c_fg / (d_full * d_gate))
            r[f"{a}{b}"] = {"q_tot": d_full, "e_q_tot": np.sqrt(v_full), "d_gate": d_gate, "R": R, "e_R": e_R}
    ris_qtot[lab] = r

# --- LED: stessa R del Gain (rapporto delle cariche medie), finestra piena e accorciata ---
for lab in ris_qtot:
    nome = f"Test_SiPM_{lab.replace('.', '-')}V_4LEDampl_330Hz_ExtTriggerLED_0000_750Ms_80PT.h5.gz"
    percorso = os.path.join(DIR_LED, nome)
    if not os.path.exists(percorso):
        print(f"[{DESCRIZIONE[lab]}] manca la run col LED {nome}")
        continue
    wf_led, dt_led = leggi_canale(percorso, CHANNEL, MAX_EVENTI_LED)
    if not np.isclose(dt_led, SAMPLING_TIME):
        raise ValueError(f"{nome}: SamplingTime {dt_led} diverso dal buio ({SAMPLING_TIME})")
    wf_led = filtra_waveform(wf_led)
    qg = compute_charge(wf_led, GATE_LED_NS, BASELINE_LED_NS)
    for chiave, fin in (("R_led_piena", FINESTRA_LED_PIENA_NS), ("R_led_corta", FINESTRA_LED_CORTA_NS)):
        qf = compute_charge(wf_led, fin, BASELINE_LED_NS)
        cv = np.cov(qf, qg) / len(qg)
        R = qf.mean() / qg.mean()
        e_R = abs(R) * np.sqrt(cv[0, 0] / qf.mean() ** 2 + cv[1, 1] / qg.mean() ** 2 - 2 * cv[0, 1] / (qf.mean() * qg.mean()))
        ris_qtot[lab][chiave] = (R, e_R)
    ris_qtot[lab]["wf_led_media"] = wf_led.mean(axis=0) - wf_led[:, (t >= BASELINE_LED_NS[0]) & (t <= BASELINE_LED_NS[1])].mean()
    del wf_led

print(f"Finestra lunga: buio {FINESTRA_BUIO_NS[0]:.0f}-{FINESTRA_BUIO_NS[1]:.0f} ns ({durata_dopo_salita:.0f} ns dopo la salita); "
      f"LED piena {FINESTRA_LED_PIENA_NS[0]:.0f}-{FINESTRA_LED_PIENA_NS[1]:.0f}, corta {FINESTRA_LED_CORTA_NS[0]:.0f}-{FINESTRA_LED_CORTA_NS[1]:.0f} ns")
print(f"{'run':>8} {'R buio 2-1':>14} {'R buio 3-2':>14} {'R LED corta':>14} {'R LED piena':>14} "
      f"{'Q_pe_tot buio 2-1':>18} {'Q_pe_tot LED corta':>19}")
for lab, r in ris_qtot.items():
    cella = lambda k: f"{r[k]['R']:6.3f} ± {r[k]['e_R']:5.3f}" if k in r else f"{'-':>14}"
    led = lambda k: f"{r[k][0]:6.3f} ± {r[k][1]:5.3f}" if k in r else f"{'-':>14}"
    qt_b = f"{r['12']['q_tot']:8.0f} ± {r['12']['e_q_tot']:4.0f}" if "12" in r else f"{'-':>18}"
    qt_l = f"{r['R_led_corta'][0] * QPE_LED[lab]:12.0f}" if "R_led_corta" in r and lab in QPE_LED else f"{'-':>19}"
    print(f"{DESCRIZIONE[lab]:>8} {cella('12')} {cella('23')} {led('R_led_corta')} {led('R_led_piena')} {qt_b:>18} {qt_l:>19}")

# --- Forma: una cella (2 p.e. - 1 p.e. medi, buio) contro la media del LED, normalizzate al massimo ---
lab_f = [l for l in ris_qtot if mostra(l) and "wf_led_media" in ris_qtot[l]]
for lab in lab_f:
    q_gate = charge_gain2[lab]
    medie_wf = {}
    for f in finestre_gain2[lab]:
        if f["n"] in (1, 2, 3):
            sel = (q_gate >= f["a"]) & (q_gate < f["b"])
            w_sel = wf_analisi[lab][sel].astype(np.float64)
            medie_wf[f["n"]] = w_sel.mean(axis=0) - w_sel[:, (t >= BASELINE_NS[0]) & (t <= BASELINE_NS[1])].mean()
    fig, ax = plt.subplots(figsize=(14, 5))
    for (a, b), col in (((1, 2), "k"), ((2, 3), "tab:orange")):
        if a in medie_wf and b in medie_wf:
            d = medie_wf[b] - medie_wf[a]
            ax.plot(t - SALITA_BUIO_NS, d / d.max(), color=col, label=f"dark: one cell ({b} p.e. - {a} p.e. mean)")
    led = ris_qtot[lab]["wf_led_media"]
    ax.plot(t - SALITA_LED_NS, led / led.max(), color="tab:blue", label="LED: mean waveform")
    ax.axvspan(GATE_LED_NS[0] - SALITA_LED_NS, GATE_LED_NS[1] - SALITA_LED_NS, color="orange", alpha=0.2, label="gate")
    ax.axvline(durata_dopo_salita, color="gray", ls=":", label="end of the dark trace")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlim(-100, 1150)
    ax.set_xlabel("Time from the rise [ns]"); ax.set_ylabel("Normalised to the maximum")
    ax.set_title(f"{DESCRIZIONE[lab]} — single-cell response vs LED mean pulse")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
